# Day 78 — Tensors

This notebook covers PyTorch Tensors, Tensor Metadata (`shape`, `dtype`, `device`), Matrix Multiplication & Broadcasting, Hardware Acceleration, and Safe NumPy Interoperability (`.detach().cpu().numpy()`)

## Validation goals
- Understand tensor dimensionality across 0D scalars, 1D vectors, 2D matrices, and 3D+ higher-order structures and their differences from NumPy arrays
- Create and inspect PyTorch tensors while tracking `shape`, `dtype` (`torch.float32`), and `device` (`cpu` or `cuda`)
- Perform matrix multiplication (`@`), broadcasting, and shape manipulation (`.reshape(...)`, `.unsqueeze(dim)`) across batch tensors
- Manage device placement with `.to(device)` and convert tensors to NumPy safely via `.detach().cpu().numpy()`

### 1. What Is a PyTorch Tensor?

A tensor is a multi-dimensional array optimized for numerical computing and automatic differentiation. If you are already comfortable with numpy.ndarray, tensors will feel immediately familiar:
- 0D Tensor (Scalar): A single number, such as a loss value (torch.tensor(0.42)).
- 1D Tensor (Vector): A sequence of values, such as a feature vector for one retail item.
- 2D Tensor (Matrix): Tabular data or a batch of samples with shape (batch_size, features).
- 3D+ Tensors: Higher-order data such as sequences (batch, seq_len, features) or images (batch, channels, height, width).

**Tensors vs. NumPy Arrays**

While both represent grids of numbers, PyTorch tensors add two major capabilities required for deep learning:
- Hardware Acceleration: Tensors can live on a CPU or GPU (CUDA) and execute parallelized matrix operations on accelerator hardware.
- Autograd Tracking: Tensors can record operations (requires_grad=True) so PyTorch can automatically compute gradients during backpropagation.

### 2. Creating and Inspecting Tensors

When working with tensors in PyTorch, always pay explicit attention to three attributes:
- shape: The dimensions along each axis.
- dtype: The data type (e.g., torch.float32, torch.int64, torch.bool). Neural network weights and floating-point computations typically use float32.
- device: Where the tensor resides in memory (cpu or cuda).

In [ ]:
import numpy as np
import torch

# 1. Creating tensors directly from Python lists or NumPy arrays
x_list = torch.tensor([[10.0, 20.0], [30.0, 40.0]], dtype=torch.float32)

np_array = np.array([[1.5, 2.5], [3.5, 4.5]], dtype=np.float32)
x_from_np = torch.from_numpy(np_array)

# 2. Common factory constructors
zeros = torch.zeros((2, 3), dtype=torch.float32)
ones = torch.ones((2, 3), dtype=torch.float32)
rand_normal = torch.randn((2, 3), dtype=torch.float32)

# 3. Inspecting tensor metadata
print("--- Day 78: Tensor Metadata ---")
print(f"x_list shape:  {x_list.shape}")
print(f"x_list dtype:  {x_list.dtype}")
print(f"x_list device: {x_list.device}\n")
print("Random normal tensor:\n", rand_normal)

### 3. Tensor Operations, Broadcasting, and Reshaping

PyTorch supports element-wise arithmetic, matrix multiplication, reshaping, and broadcasting in ways that closely mirror NumPy:
- Element-wise operations: +, -, *, / operate element-by-element across matching or broadcastable shapes.
- Matrix multiplication: Use @ or torch.matmul(a, b) for linear algebra products ((N, D) @ (D, M) -> (N, M)).
- Shape manipulation: Use .reshape(...) or .view(...) to change dimensions without altering underlying data order, and `.unsqueeze(dim)` / `.squeeze(dim)` to add or remove size-1 dimensions.

In [ ]:
import torch

# Simulate a batch of 3 samples with 4 features, passing through a layer with 2 outputs
X = torch.randn((3, 4), dtype=torch.float32)
W = torch.randn((4, 2), dtype=torch.float32)
b = torch.zeros((1, 2), dtype=torch.float32)

# Linear layer computation using matrix multiplication (@) and bias broadcasting
Z = X @ W + b

# Reshaping and dimension inspection
flat_Z = Z.reshape(-1)
unsqueezed_vec = torch.tensor([1.0, 2.0, 3.0]).unsqueeze(1)  # Shape: (3, 1)

print("--- Day 78: Tensor Operations ---")
print(f"X shape: {X.shape}, W shape: {W.shape}, Z shape: {Z.shape}")
print(f"Flattened Z shape: {flat_Z.shape}")
print(f"Unsqueezed vector shape: {unsqueezed_vec.shape}")

### 4. Device Management and NumPy Interoperability

In real training scripts, models and tensors must reside on the same device—mixing a CPU tensor with a GPU tensor raises a runtime error. A clean pattern is defining a target device upfront and moving tensors explicitly with .`to(device)`.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tensor_a = torch.tensor([2.0, 4.0, 6.0], dtype=torch.float32).to(device)
tensor_b = torch.tensor([1.0, 3.0, 5.0], dtype=torch.float32).to(device)

result = (tensor_a + tensor_b) * 0.5

# Move back to CPU before converting to a NumPy array
result_np = result.detach().cpu().numpy()

print("--- Day 78: Device & NumPy Interop ---")
print(f"Active device: {device}")
print(f"Result tensor: {result}")
print(f"NumPy array type: {type(result_np)}, values: {result_np}")

Notice `.detach().cpu().numpy()`: calling `.detach()` detaches the tensor from any computation graph, `.cpu()` ensures it is on host memory, and .numpy() converts it cleanly to NumPy.

### Guided Question:

Beyond syntax similarity to NumPy arrays, what two capabilities make PyTorch tensors essential for training deep neural networks?

### Answer:

Beyond feeling familiar like NumPy `ndarrays`, PyTorch tensors are essential for deep learning because of hardware acceleration (especially GPU execution) and automatic differentiation (`autograd`).

**1. Hardware Acceleration (CPU, GPU, and Beyond)**

NumPy arrays live in standard system RAM and execute operations on the CPU. Training deep neural networks requires millions or billions of matrix multiplications and element-wise operations, which CPUs handle slowly compared to GPUs designed for massive parallel throughput.
- Explicit device placement: A PyTorch tensor can live on a CPU (`device="cpu"`) or be moved to a GPU (`device="cuda"` for NVIDIA GPUs, or `device="mps"` on Apple Silicon) using `.to(device)` or `.cuda().`
- Same code, faster hardware: Once tensors are on the GPU, tensor operations run in parallel across thousands of GPU cores without rewriting your mathematical logic.
- Memory trade-off: Moving data between CPU host memory and GPU device memory has transfer overhead, so batches are typically moved to the GPU and kept there through the forward and backward pass.

**2. Automatic Differentiation (`autograd`)**

Training a neural network requires computing the gradient of a loss function with respect to every trainable parameter (weights and biases) so gradient descent knows which direction to update them. Writing derivatives by hand or numerically with finite differences does not scale to deep models.
- Dynamic computation graph: When a tensor has `requires_grad=True`, PyTorch records the operations performed on it in a directed acyclic graph (DAG) as the forward pass runs.
- Reverse-mode backprop: Calling `loss.backward() `traverses that graph backward using the chain rule and stores the resulting gradients in each parameter tensor's `.grad` attribute.
- Inference control: During evaluation or inference, wrapping code in with torch.`no_grad()`: disables graph tracking to save memory and speed up execution.

In short: GPUs compute the forward pass fast, and autograd computes the gradients for learning automatically.

### Guided Question:

If an input batch tensor X has shape (64, 8) and a weight matrix W has shape (8, 16), what is the shape of X @ W, and why must the inner dimensions match?

### Answer:

The resulting tensor X @ W has shape (64, 16).

How the Output Shape WorksIn matrix multiplication between two 2D tensors of shapes $(M, K)$ and $(K, N)$, the inner dimensions ($K$) cancel out and the outer dimensions remain, giving an output shape of $(M, N)$:$$\underbrace{(64, 8)}_{X} \times \underbrace{(8, 16)}_{W} \longrightarrow (64, 16)$$

**In deep learning terms:**

- 64 is your batch size (64 individual data samples).
- 8 is the number of input features per sample.
- 16 is the number of output features (neurons) in that layer.
- The output (`64, 16`) means each of the 64 samples now has 16 newly computed features.

**Why the Inner Dimensions Must Match**

Matrix multiplication works by taking the dot product of each row in X with each column in W.
- One weight per input feature: A single row of X contains 8 numbers (one sample's features). A single column of W contains 8 numbers (the weights for one output neuron). To multiply them element-by-element and sum them up, both vectors must have the exact same length (8).
- Dimensional mismatch causes an error: If X has 8 features per sample and W expects 10 features per neuron, there is no weight for the 9th and 10th positions—so PyTorch raises a RuntimeError for shape mismatch.

### Guided Question:

Why is `.detach().cpu().numpy() `a safer pattern for converting a tensor to NumPy than calling `.numpy()` directly on a tensor?

### Answer:

Calling `.numpy()` directly on a tensor fails in two common deep learning situations: when the tensor is attached to an autograd computation graph with gradients enabled (`requires_grad=True`) or when it resides on a non-CPU device (such as a GPU on cuda). Using `.detach().cpu().numpy()` avoids both pitfalls cleanly.

**1. `.detach()` — Unlinking from the Computation Graph**

NumPy arrays do not support PyTorch autograd or gradient tracking. If a tensor has `requires_grad=True` and is still connected to a computation graph, `calling .numpy()` directly raises a RuntimeError because PyTorch will not silently discard gradient tracking on an active graph node.
- What `.detach()` does: It returns a new tensor view that shares the underlying data buffer but is detached from the computation graph (`requires_grad=False`).
- When it matters: Whenever you inspect intermediate activations, predictions, or loss values produced during a forward pass where gradients are tracked.

**2. `.cpu()` — Moving Data to Host RAM**

NumPy operates on standard CPU memory (host RAM) and cannot directly read memory located on a GPU (cuda or mps). Calling `.numpy()` directly on a GPU tensor raises a TypeError (can't convert cuda:... device type tensor to numpy).
- What `.cpu()` does: It copies the tensor onto CPU memory if it is currently on a GPU (and is a no-op if it is already on the CPU).
- Memory note: Because `.cpu()` may allocate a copy in host RAM, you should avoid calling it inside tight training loops unless you specifically need the values outside PyTorch (such as for logging, plotting, or Scikit-learn metrics).

**3. `.numpy()` — Final Conversion to ndarray**

Once the tensor is detached from gradient tracking and placed on the CPU, .`numpy() `exposes the data as a NumPy ndarray.
- Shared memory caveat: If the tensor was already on the CPU and detached, the resulting NumPy array typically shares the same underlying memory buffer—meaning mutating one in place can mutate the other. If you need a completely independent copy, call `.copy()` afterward.

In short: `.detach()` drops gradient tracking, `.cpu()` moves the data to CPU memory, and `.numpy()` converts it to a NumPy array without hitting device or autograd errors.